## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


## 2. Load Data

In [ ]:

train = pd.read_csv(r'C:\kaggle\train.csv')
test  = pd.read_csv(r'C:\kaggle\test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)


## 3. EDA

In [ ]:
train.info()


In [ ]:
train.describe()


In [ ]:
train.isnull().sum()


In [ ]:
sns.heatmap(train.select_dtypes(include=[np.number]).corr(),
            annot=True, fmt='.2f', cmap='coolwarm')
plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(data=train, x='PitStop', kde=True)
plt.show()


## 4. Handle Missing Values

In [ ]:
# Fill PitStop nulls with 0
train['PitStop'] = train['PitStop'].fillna(0.0)
test['PitStop']  = test['PitStop'].fillna(0.0)

print("Train nulls after fill:")
print(train.isnull().sum())


## 5. Feature Engineering

In [ ]:

for df in [train, test]:
    df['TyreLife_squared']     = df['TyreLife'] ** 2
    df['LapTime_x_TyreLife']   = df['LapTime (s)'] * df['TyreLife']
    df['Degradation_per_lap']  = df['Cumulative_Degradation'] / (df['LapNumber'] + 1)

    df['TyreAge_ratio']        = df['TyreLife'] / (df['LapNumber'] + 1)
    df['Gap_x_Position']       = df['GapToLeader'] * df['Position']

print("New columns added:", ['TyreLife_squared','LapTime_x_TyreLife',
                              'Degradation_per_lap','TyreAge_ratio','Gap_x_Position'])


## 6. Define Features and Target

In [ ]:
x      = train.drop(['PitNextLap', 'id'], axis=1)
y      = train['PitNextLap']
x_test = test.drop(['id'], axis=1)

print("x shape      :", x.shape)
print("x_test shape :", x_test.shape)
print("Target balance:\n", y.value_counts())

ratio = (y == 0).sum() / (y == 1).sum()
print(f"\nClass ratio (scale_pos_weight): {ratio:.2f}")


## 7. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

x_train_raw, x_val_raw, y_train, y_val = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("x_train_raw:", x_train_raw.shape)
print("x_val_raw  :", x_val_raw.shape)


## 8. Preprocessing (OrdinalEncoder for LightGBM)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

cat_features = ['Driver', 'Compound', 'Race']
num_features = [col for col in x.columns if col not in cat_features]

lgbm_preprocessor = ColumnTransformer([
    ("OrdinalEncoder", OrdinalEncoder(handle_unknown='use_encoded_value',
                                       unknown_value=-1), cat_features),
    ("passthrough",    "passthrough",                    num_features)
])

x_encoded_train = lgbm_preprocessor.fit_transform(x_train_raw)
x_encoded_val   = lgbm_preprocessor.transform(x_val_raw)
x_encoded_test  = lgbm_preprocessor.transform(x_test)

print("x_encoded_train:", x_encoded_train.shape)
print("x_encoded_val  :", x_encoded_val.shape)
print("x_encoded_test :", x_encoded_test.shape)


## 9. Optuna Hyperparameter Tuning

In [ ]:
!pip install optuna -q


In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'max_bin'           : trial.suggest_int('max_bin', 255, 512),
        'n_estimators'      : trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.05),
        'num_leaves'        : trial.suggest_int('num_leaves', 63, 300),
        'max_depth'         : trial.suggest_int('max_depth', 6, 12),
        'boosting_type'     : trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0.0, 0.5),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0.0, 0.5),
        'scale_pos_weight'  : ratio,
        'device'            : 'cpu',
        'random_state'      : 42,
        'verbose'           : -1,
        'n_jobs'            : -1,
    }

    model = LGBMClassifier(**params)


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, x_encoded_train, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\nBest CV ROC-AUC:", study.best_value)
print("Best Params    :", study.best_params)


## 10. Train Final Model
>  **NEW**: Trains on full training data with early stopping on validation set.

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

best_params = study.best_params.copy()


n_est = best_params.pop('n_estimators', 2000)

best_lgbm = LGBMClassifier(
    **best_params,
    n_estimators=3000,       
    scale_pos_weight=ratio,
    device='cpu',
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)


best_lgbm.fit(
    x_encoded_train, y_train,
    eval_set=[(x_encoded_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=50, verbose=False),
        log_evaluation(period=100)
    ]
)

print(f"Best iteration: {best_lgbm.best_iteration_}")


## 11. Evaluate on Validation Set

In [ ]:
from sklearn.metrics import (roc_auc_score, accuracy_score,
                              f1_score, classification_report)

y_pred_proba = best_lgbm.predict_proba(x_encoded_val)[:, 1]
y_pred       = best_lgbm.predict(x_encoded_val)

print(f"ROC-AUC   : {roc_auc_score(y_val, y_pred_proba):.4f}")
print(f"Accuracy  : {accuracy_score(y_val, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_val, y_pred):.4f}")
print()
print(classification_report(y_val, y_pred))


## 12. Feature Importance

In [ ]:
feature_names = cat_features + num_features
importances   = best_lgbm.feature_importances_

feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(x='importance', y='feature', data=feat_imp, palette='viridis')
plt.title('Top 20 Feature Importances (LightGBM)')
plt.tight_layout()
plt.show()


## 13. Generate Submission

In [ ]:
final_preds = best_lgbm.predict_proba(x_encoded_test)[:, 1]

submission = pd.DataFrame({
    'id'        : test['id'],
    'PitNextLap': final_preds
})

print("Rows   :", len(submission))
print("Preview:\n", submission.head())

submission.to_csv('submission.csv', index=False)
print("\n submission.csv saved!")
